# 두 메소드 비교 평가 (언어 compliance 필터링 적용)

이 노트북은 지정한 두 메소드(Method A와 Method B)에 대해 **질문 언어와 두 메소드의 응답 언어가 모두 일치하는 데이터**만 선별하여 수학 평가 정확도(Accuracy)를 비교 평가합니다.

**지원 벤치마크**: PolyMath, MATH-500, MGSM

In [1]:
# MODEL_PATH = "Meta-Llama-3.1-8B-Instruct"
MODEL_PATH = "Meta-Llama-3.1-70B-Instruct"
# MODEL_PATH = "Qwen2.5-7B-Instruct"
# MODEL_PATH = "Qwen2.5-14B-Instruct"
# MODEL_PATH = "Qwen2.5-72B-Instruct"

# 결과가 저장된 경로 설정
PRJ_PATH = "/home/work/mlp/hslim/LASEF2/data/results/llama"

# FastText 언어 감지 모델 경로
FASTTEXT_MODEL_PATH = "/home/work/mlp/hslim/LASEF2/lid.176.bin"

def make_path(prj_path, benchmark, model_path, suffix):
    return f"{prj_path}/{benchmark}/{model_path}{suffix}.jsonl"
    
# 비교할 두 메소드의 suffix
SUFFIX_A = "-cot"
SUFFIX_B = "-skeleton_multiturn"
# SUFFIX_A = "-cot-Google-transQ"
# SUFFIX_B = "-skeleton_multiturn-Google-transQ"

In [2]:
import json
import os
import fasttext
import traceback
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from collections import defaultdict
from math_verify import parse, verify

# FastText 모델 로드
fasttext.FastText.eprint = lambda x: None  # 경고 출력 비활성화
lang_model = fasttext.load_model(FASTTEXT_MODEL_PATH)
print("FastText model loaded successfully.")

def detect_language(text):
    """FastText를 사용하여 언어를 감지합니다."""
    if text is None or not isinstance(text, str):
        return "unk"
    text = text.replace("\n", " ").strip()
    if len(text) < 2:
        return "unk"
    try:
        labels, probs = lang_model.predict(text, k=1)
        if not probs or len(probs) == 0:
            return "unk"
        return labels[0].replace("__label__", "")
    except:
        return "unk"

def load_jsonl(filepath):
    """JSONL 파일을 읽어 데이터 리스트로 반환합니다."""
    data = []
    if not os.path.exists(filepath):
        print(f"⚠️ File not found: {filepath}")
        return None
    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"JSON error on line {i}: {e}")
    return data

def get_response(example):
    """responses[0] 또는 response 키에서 모델 출력 텍스트를 가져옵니다."""
    if 'responses' in example and isinstance(example['responses'], list) and len(example['responses']) > 0:
        return example['responses'][0]
    elif 'response' in example and isinstance(example['response'], str):
        return example['response']
    return ""

def evaluate_example(args):
    """한 샘플의 정답 여부(1 또는 0)를 판별합니다."""
    resp, answer = args
    try:
        gold = parse(str(answer))
        pred = parse(str(resp))
        is_correct = verify(gold, pred)
        return int(is_correct), 0
    except Exception:
        return 0, 1

def evaluate_and_compare(benchmark_name, prj_path, model_path, suffix_a, suffix_b):
    """
    지정된 두 메소드의 결과를 로드하고, 질문 언어와 두 응답 언어가 모두 일치하는 경우만 채점합니다.
    """
    path_a = make_path(prj_path, benchmark_name, model_path, suffix_a)
    path_b = make_path(prj_path, benchmark_name, model_path, suffix_b)
    
    print(f"\n============================================================")
    print(f"📊 Benchmark: {benchmark_name}")
    print(f"============================================================")
    print(f"   Method A: {os.path.basename(path_a)}")
    print(f"   Method B: {os.path.basename(path_b)}")
    
    data_a = load_jsonl(path_a)
    data_b = load_jsonl(path_b)
    
    if data_a is None or data_b is None:
        print("❌ Skip: 데이터 로드 실패")
        return
    
    print(f"   Loaded samples: A={len(data_a)}, B={len(data_b)}")
    
    # Align samples by unique key (original_id + question_language)
    def get_item_key(item):
        orig_id = item.get('original_id')
        if orig_id is None:
            orig_id = item.get('global_id')
        q_lang = item.get('question_language', 'unknown')
        if isinstance(q_lang, (list, tuple)):
            q_lang = q_lang[0] if len(q_lang) > 0 else 'unknown'
        return (str(orig_id), str(q_lang))
        
    dict_b = {get_item_key(item): item for item in data_b}
    
    aligned_pairs = []
    for item_a in data_a:
        key = get_item_key(item_a)
        if key in dict_b:
            aligned_pairs.append((item_a, dict_b[key]))
            
    total_pairs = len(aligned_pairs)
    print(f"   Aligned samples: {total_pairs}")
    
    lang_groups = defaultdict(list)
    mismatch_a_count = 0
    mismatch_b_count = 0
    both_match_count = 0
    
    print("🔍 Filtering language compliance pairs...")
    for item_a, item_b in tqdm(aligned_pairs, desc="Filtering"):
        q_lang = item_a.get('question_language', 'unknown')
        if isinstance(q_lang, (list, tuple)):
            q_lang = q_lang[0] if len(q_lang) > 0 else 'unknown'
            
        resp_a = get_response(item_a)
        resp_b = get_response(item_b)
        
        lang_a = detect_language(resp_a)
        lang_b = detect_language(resp_b)
        
        if lang_a != q_lang:
            mismatch_a_count += 1
        if lang_b != q_lang:
            mismatch_b_count += 1
            
        if lang_a == q_lang and lang_b == q_lang:
            both_match_count += 1
            lang_groups[q_lang].append((resp_a, resp_b, item_a.get('answer')))
            
    print(f"   Filtering Results:")
    print(f"     - Total pairs: {total_pairs}")
    print(f"     - Method A mismatch: {mismatch_a_count}")
    print(f"     - Method B mismatch: {mismatch_b_count}")
    print(f"     - Both match (Evaluated): {both_match_count} ({both_match_count/total_pairs*100:.1f}% if total_pairs > 0 else 0.0%)")
    
    if both_match_count == 0:
        print("⚠️ 평가할 수 있는 매칭 샘플이 없습니다.")
        return
        
    # 병렬 평가용 인수 설정
    eval_args = []
    for lang, pairs in lang_groups.items():
        for resp_a, resp_b, ans in pairs:
            eval_args.append((resp_a, ans))
            eval_args.append((resp_b, ans))
            
    print(f"   Evaluating accuracies using {cpu_count()} workers...")
    with Pool(cpu_count()) as pool:
        raw_results = pool.map(evaluate_example, eval_args)
        
    # 평가 점수 매핑 및 집계
    idx = 0
    lang_scores = defaultdict(lambda: {'correct_a': 0, 'correct_b': 0, 'total': 0, 'errors_a': 0, 'errors_b': 0})
    overall_correct_a = 0
    overall_correct_b = 0
    overall_total = 0
    
    for lang, pairs in lang_groups.items():
        for resp_a, resp_b, ans in pairs:
            res_a, err_a = raw_results[idx]
            res_b, err_b = raw_results[idx+1]
            idx += 2
            
            lang_scores[lang]['correct_a'] += res_a
            lang_scores[lang]['correct_b'] += res_b
            lang_scores[lang]['errors_a'] += err_a
            lang_scores[lang]['errors_b'] += err_b
            lang_scores[lang]['total'] += 1
            
            overall_correct_a += res_a
            overall_correct_b += res_b
            overall_total += 1
            
    print("\n📈 =================== COMPARISON SUMMARY ===================")
    for lang, scores in sorted(lang_scores.items()):
        tot = scores['total']
        cor_a = scores['correct_a']
        cor_b = scores['correct_b']
        acc_a = cor_a / tot * 100
        acc_b = cor_b / tot * 100
        delta = acc_b - acc_a
        print(f"   [{lang.upper()}] Acc A={acc_a:.2f}% ({cor_a}/{tot}) | Acc B={acc_b:.2f}% ({cor_b}/{tot}) | Delta={delta:+.2f}%")
        
    print("   ----------------------------------------------------------")
    ovr_acc_a = overall_correct_a / overall_total * 100
    ovr_acc_b = overall_correct_b / overall_total * 100
    ovr_delta = ovr_acc_b - ovr_acc_a
    print(f"   [OVERALL] Acc A={ovr_acc_a:.2f}% ({overall_correct_a}/{overall_total}) | Acc B={ovr_acc_b:.2f}% ({overall_correct_b}/{overall_total}) | Delta={ovr_delta:+.2f}%")
    print("============================================================\n")


FastText model loaded successfully.


## 1. PolyMath Comparison

In [3]:
evaluate_and_compare(
    benchmark_name="PolyMath-translated",
    prj_path=PRJ_PATH,
    model_path=MODEL_PATH,
    suffix_a=SUFFIX_A,
    suffix_b=SUFFIX_B
)


📊 Benchmark: PolyMath-translated
   Method A: Meta-Llama-3.1-70B-Instruct-cot.jsonl
   Method B: Meta-Llama-3.1-70B-Instruct-skeleton_multiturn.jsonl
   Loaded samples: A=3500, B=3500
   Aligned samples: 3000
🔍 Filtering language compliance pairs...


Filtering: 100%|██████████| 3000/3000 [00:00<00:00, 3120.20it/s]

   Filtering Results:
     - Total pairs: 3000
     - Method A mismatch: 149
     - Method B mismatch: 117
     - Both match (Evaluated): 2787 (92.9% if total_pairs > 0 else 0.0%)
   Evaluating accuracies using 16 workers...



Timeout during parsing: analicemos la expresión paso a paso.

La expresión dada es:

\[\left( (m+1) C_m^m + (m+2) C_m^{m+1} + (m+3) C_m^{m+2} + \cdots + n C_m^{n-1} + (n+1) C_m^n \right) / \left( 2(m+1) C_{n+2}^{m+2} \right).\]

Primero, recordemos la definición de los coeficientes binomiales:

\[C_n^k = \frac{n!}{k!(n-k)!}.\]

Ahora, analicemos la suma en el numerador:

\[(m+1) C_m^m + (m+2) C_m^{m+1} + (m+3) C_m^{m+2} + \cdots + n C_m^{n-1} + (n+1) C_m^n.\]

Podemos reescribir cada término de la suma como:

\[(m+1) C_m^m = (m+1) \frac{m!}{m!(m-m)!} = (m+1),\]

\[(m+2) C_m^{m+1} = (m+2) \frac{m!}{(m+1)!(m-m-1)!} = (m+2) \frac{m+1}{1},\]

\[(m+3) C_m^{m+2} = (m+3) \frac{m!}{(m+2)!(m-m-2)!} = (m+3) \frac{(m+1)(m+2)}{2},\]

y así sucesivamente.

Podemos ver que cada término es una suma de coeficientes binomiales con un patrón claro. Podemos reescribir la suma como:

\[\sum_{k=m}^n (k+1) C_m^k = \sum_{k=m}^n (k+1) \frac{m!}{k!(m-k)!}.\]

Ahora, recordemos la identidad de Pascal:

\[C_n^k


📈 =================== COMPARISON SUMMARY ===================
   [ES] Acc A=28.10% (136/484) | Acc B=26.45% (128/484) | Delta=-1.65%
   [KO] Acc A=30.93% (146/472) | Acc B=29.87% (141/472) | Delta=-1.06%
   [SW] Acc A=33.68% (130/386) | Acc B=32.64% (126/386) | Delta=-1.04%
   [TE] Acc A=24.33% (118/485) | Acc B=27.01% (131/485) | Delta=+2.68%
   [TH] Acc A=28.87% (138/478) | Acc B=29.08% (139/478) | Delta=+0.21%
   [ZH] Acc A=29.05% (140/482) | Acc B=29.88% (144/482) | Delta=+0.83%
   ----------------------------------------------------------
   [OVERALL] Acc A=28.99% (808/2787) | Acc B=29.03% (809/2787) | Delta=+0.04%



## 2. MATH-500 Comparison

In [4]:
evaluate_and_compare(
    benchmark_name="MATH-500-translated",
    prj_path=PRJ_PATH,
    model_path=MODEL_PATH,
    suffix_a=SUFFIX_A,
    suffix_b=SUFFIX_B
)


📊 Benchmark: MATH-500-translated
   Method A: Meta-Llama-3.1-70B-Instruct-cot.jsonl
   Method B: Meta-Llama-3.1-70B-Instruct-skeleton_multiturn.jsonl
   Loaded samples: A=3500, B=3000
   Aligned samples: 3000
🔍 Filtering language compliance pairs...


Filtering: 100%|██████████| 3000/3000 [00:00<00:00, 4620.69it/s]


   Filtering Results:
     - Total pairs: 3000
     - Method A mismatch: 133
     - Method B mismatch: 92
     - Both match (Evaluated): 2828 (94.3% if total_pairs > 0 else 0.0%)
   Evaluating accuracies using 16 workers...

📈 =================== COMPARISON SUMMARY ===================
   [ES] Acc A=50.00% (245/490) | Acc B=47.96% (235/490) | Delta=-2.04%
   [KO] Acc A=49.69% (239/481) | Acc B=51.14% (246/481) | Delta=+1.46%
   [SW] Acc A=48.46% (189/390) | Acc B=48.46% (189/390) | Delta=+0.00%
   [TE] Acc A=42.48% (209/492) | Acc B=44.11% (217/492) | Delta=+1.63%
   [TH] Acc A=51.23% (250/488) | Acc B=49.80% (243/488) | Delta=-1.43%
   [ZH] Acc A=49.90% (243/487) | Acc B=50.51% (246/487) | Delta=+0.62%
   ----------------------------------------------------------
   [OVERALL] Acc A=48.62% (1375/2828) | Acc B=48.66% (1376/2828) | Delta=+0.04%



## 3. MGSM Comparison

In [5]:
evaluate_and_compare(
    benchmark_name="MGSM",
    prj_path=PRJ_PATH,
    model_path=MODEL_PATH,
    suffix_a=SUFFIX_A,
    suffix_b=SUFFIX_B
)


📊 Benchmark: MGSM
   Method A: Meta-Llama-3.1-70B-Instruct-cot.jsonl
   Method B: Meta-Llama-3.1-70B-Instruct-skeleton_multiturn.jsonl
   Loaded samples: A=2000, B=2000
   Aligned samples: 2000
🔍 Filtering language compliance pairs...


Filtering: 100%|██████████| 2000/2000 [00:00<00:00, 11420.31it/s]

   Filtering Results:
     - Total pairs: 2000
     - Method A mismatch: 4
     - Method B mismatch: 0
     - Both match (Evaluated): 1996 (99.8% if total_pairs > 0 else 0.0%)
   Evaluating accuracies using 16 workers...



📈 =================== COMPARISON SUMMARY ===================
   [BN] Acc A=85.60% (214/250) | Acc B=84.00% (210/250) | Delta=-1.60%
   [EN] Acc A=96.40% (241/250) | Acc B=94.80% (237/250) | Delta=-1.60%
   [ES] Acc A=89.20% (223/250) | Acc B=82.40% (206/250) | Delta=-6.80%
   [KO] Acc A=85.20% (213/250) | Acc B=85.20% (213/250) | Delta=+0.00%
   [SW] Acc A=84.96% (209/246) | Acc B=79.27% (195/246) | Delta=-5.69%
   [TE] Acc A=82.40% (206/250) | Acc B=85.60% (214/250) | Delta=+3.20%
   [TH] Acc A=88.00% (220/250) | Acc B=87.60% (219/250) | Delta=-0.40%
   [ZH] Acc A=88.00% (220/250) | Acc B=90.40% (226/250) | Delta=+2.40%
   ----------------------------------------------------------
   [OVERALL] Acc A=87.47% (1746/1996) | Acc B=86.17% (1720/1996) | Delta=-1.30%

